# BizLens v2.2.12 - PCA & Clustering (Comprehensive)

Dimensionality reduction and unsupervised learning using **BizLens + scikit-learn**.
We will use the classic Iris and Penguins datasets.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens"])
print("✅ BizLens v2.2.13 ready for PCA & Clustering!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
# bl.set_profiling(True)

# ── Optional: Use a BizLens built-in dataset ─────────────────────────────
# df = bl.load_dataset('penguins')  # cluster penguins by bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
# df = bl.load_dataset('iris')  # classic PCA + clustering — 4 numeric features, 3 species
#
# After loading, explore with:
# print(df.shape)         # rows × columns
# print(df.dtypes)        # column types
# print(df.head())        # first 5 rows
#
# ── Export / Save Results ─────────────────────────────────────────────────
# df['cluster'] = labels; df.to_csv('clustered_output.csv', index=False)
# df.to_excel('output.xlsx', index=False)  # Save as Excel
# df.to_json('output.json', orient='records', indent=2)  # Save as JSON

## 1. Load Dataset (Iris & Penguins)

In [ ]:
iris = bl.load_dataset("iris")
penguins = bl.load_dataset("penguins")

print("Iris dataset:")
bl.describe(iris)

print("\nPenguins dataset:")
bl.describe(penguins)

## 2. Data Quality Check with BizLens

In [ ]:
bl.quality.data_profile(iris)
bl.quality.data_profile(penguins)

## 3. Principal Component Analysis (PCA)

In [ ]:
# Prepare numeric data
X_iris = iris.select_dtypes(include=np.number)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print(f"Explained variance by components: {explained}")
print(f"Cumulative explained variance: {cumulative}")

In [ ]:
# Scree Plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained)+1), explained, marker='o', label='Explained Variance')
plt.plot(range(1, len(explained)+1), cumulative, marker='s', label='Cumulative')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot - Iris Dataset')
plt.legend()
plt.grid(True)
plt.show()

## 4. Clustering (K-Means)

In [ ]:
# Elbow Method
inertia = []
sil_scores = []
K = range(2, 8)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_scaled, kmeans.labels_))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(K, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')

plt.subplot(1, 2, 2)
plt.plot(K, sil_scores, marker='o', color='green')
plt.title('Silhouette Score')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.show()

In [ ]:
# Final K-Means with optimal clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

iris['cluster'] = clusters
bl.tables.contingency_table(iris, 'species', 'cluster')[0]

## 5. Visualization of Clusters

In [ ]:
plt.figure(figsize=(12, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=iris['species'], style=iris['cluster'], palette='Set2')
plt.title('PCA + K-Means Clustering on Iris Dataset')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Species / Cluster')
plt.show()